# CCNY Fall 2021 Academic Calendar Scraper

This notebook scrapes the City College of New York (CCNY) Fall 2021 academic calendar. Using `requests` and `BeautifulSoup`, we will extract the calendar data from the registrar's HTML table. Then, we will turn it into a DataFrame where the index is a valid Python date, containing columns for the day of the week (`dow`) and the explanation (`text`).

## 1. Import Libraries and Fetch HTML

In [49]:
# Necessary Libraries
import requests
import pandas as pd
from bs4 import BeautifulSoup

# Fetch HTML
url = 'https://www.ccny.cuny.edu/registrar/fall'
request = requests.get(url)
soup = BeautifulSoup(request.content, 'html.parser')

## 2. Extract Data from Table

### 2.1 Inspect Table Structure

In [ ]:
# Find the table
table = soup.find('table')

# Inspect its structure
rows = table.find_all('tr')
print(f"Table has {len(rows)} rows")

# Look at the first row (header)
header_row = rows[0]
print(f"\nHeader row: {header_row.get_text(strip=True)}")

# Look at a sample data row
sample_row = rows[1]
cells = sample_row.find_all('td')
print(f"\nSample row has {len(cells)} columns:")
for i, cell in enumerate(cells):
    print(f"  Column {i}: {cell.get_text(strip=True)[:50]}")

### 2.2 Extract the Data

In [ ]:
# Find the table and extract data
table = soup.find('table')
rows = table.find_all('tr')

# Loop through rows and extract data
data = []
for row in rows[1:]:  # Skip header row
    cells = row.find_all('td')
    if len(cells) >= 3:
        date_str = cells[0].get_text(strip=True)
        dow = cells[1].get_text(strip=True)
        text = cells[2].get_text(strip=True)
        
        data.append({
            'date': date_str,
            'dow': dow,
            'text': text
        })

print(f"Extracted {len(data)} entries")
print("\nFirst 5 entries:")
for i, entry in enumerate(data[:5]):
    print(f"{i+1}. {entry['date']} | {entry['dow']} | {entry['text'][:40]}")

## 3. Create DataFrame with Date Index

In [ ]:
# Create DataFrame with date as index
dates = []
dows = []
texts = []

for entry in data:
    date_str = entry['date']
    # If date is a range, just take the first date
    if ' - ' in date_str:
        date_str = date_str.split(' - ')[0]
    # Add year 2021
    date_str = date_str + " 2021"
    dates.append(pd.to_datetime(date_str).date())
    dows.append(entry['dow'])
    texts.append(entry['text'])

# Create DataFrame
df = pd.DataFrame({'dow': dows, 'text': texts}, index=dates)
print("DataFrame created!")
print(f"Shape: {df.shape}")
print(f"Columns: {df.columns.tolist()}")
print(df)